In [8]:
# paths setzen und files auflisten

from pathlib import Path
import pandas as pd
import pyreadstat

data_dir = Path("data")
files = sorted(data_dir.rglob("*.sav"))
for f in files:
    print(f)

data\de\SC1\SC1_NEPScomp_Competency_2-0-0.sav
data\de\SC1\SC1_NEPScomp_Demography_2-0-0.sav
data\de\SC1\SC1_NEPScomp_LearningEnv_2-0-0.sav
data\de\SC1\SC1_NEPScomp_Migration_2-0-0.sav
data\de\SC1\SC1_NEPScomp_Personality_2-0-0.sav
data\de\SC1\SC1_NEPScomp_Resources_2-0-0.sav
data\de\SC1\SC1_NEPScomp_Society_2-0-0.sav
data\de\SC1\SC1_NEPScomp_States_2-0-0.sav
data\de\SC2\SC2_NEPScomp_Competency_2-0-0.sav
data\de\SC2\SC2_NEPScomp_Demography_2-0-0.sav
data\de\SC2\SC2_NEPScomp_LearningEnv_2-0-0.sav
data\de\SC2\SC2_NEPScomp_Migration_2-0-0.sav
data\de\SC2\SC2_NEPScomp_Personality_2-0-0.sav
data\de\SC2\SC2_NEPScomp_Resources_2-0-0.sav
data\de\SC2\SC2_NEPScomp_Society_2-0-0.sav
data\de\SC2\SC2_NEPScomp_States_2-0-0.sav
data\de\SC3\SC3_NEPScomp_Competency_2-0-0.sav
data\de\SC3\SC3_NEPScomp_Demography_2-0-0.sav
data\de\SC3\SC3_NEPScomp_LearningEnv_2-0-0.sav
data\de\SC3\SC3_NEPScomp_Migration_2-0-0.sav
data\de\SC3\SC3_NEPScomp_Personality_2-0-0.sav
data\de\SC3\SC3_NEPScomp_Resources_2-0-0.sav
da

In [9]:
# Metadaten und Variablennamen einer Beispieldatei auslesen
pfad = files[22]
print(pfad.name)

_, meta = pyreadstat.read_sav(pfad, metadataonly=True)
print(meta.number_rows, "Zeilen,", meta.number_columns, "Spalten")

var_info = pd.DataFrame({"var": meta.column_names, "label": meta.column_labels})
var_info

SC3_NEPScomp_Society_2-0-0.sav
108121 Zeilen, 563 Spalten


,var,label
0,ID_t,Target-ID
1,wave,Welle
2,p28430c,Unterstützung - Häufigkeit: Unterhaltung über Themen
3,p28430d,Unterstützung - Häufigkeit: Unterhaltung über Probleme
4,p34009a,"Partizipation an Hochkultur: Museum, Kunstausstellung"
...,...,...
558,th34331,"ICT: Verwendung Standardsoftware, Haupterwerbstätigkeit"
559,th34332,"ICT: Programmfunktionen Standardsoftware, Haupterwerbstätigkeit"
560,th34333,"ICT: Verwendung spezieller Software, Haupterwerbstätigkeit"
561,th34334,"ICT: Programmfunktionen spezielle Software, Haupterwerbstätigkeit"


In [10]:
# Alle Variablen und Labels aus allen Dateien auslesen

from pathlib import Path
import pandas as pd
import pyreadstat

data_dir = Path("data")
rows = []
for f in sorted(data_dir.rglob("*.sav")):
    _, meta = pyreadstat.read_sav(f, metadataonly=True)
    teile = f.stem.split("_")          # z. B. SC3_NEPScomp_Competency_1-0-0
    for var, label in zip(meta.column_names, meta.column_labels):
        rows.append({
            "kohorte": teile[0],
            "datensatz": teile[2] if len(teile) > 2 else f.stem,
            "var": var,
            "label": label,
        })

alle = pd.DataFrame(rows)
print(alle.shape)
alle.head()

(25290, 4)


,kohorte,datensatz,var,label
0,SC1,Competency,ID_t,Target-ID
1,SC1,Competency,wave,Welle
2,SC1,Competency,bd_sc3a,Zahlenspanne rückwärts: Summe
3,SC1,Competency,bd_sc3b,Zahlenspanne rückwärts: max. erreichte Ziffernspanne
4,SC1,Competency,ca_sc1,QUERSCHNITT: SON Kategorien: WLE (korrigiert)


In [16]:
# Anzeigeoptionen für Pandas einstellen
# Suche nach Variablen und Labels vereinfachen

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)

def suche(begriff, kohorte=None, quelle=None):
    """quelle='p' = Angaben der Eltern, 't' = Angaben der Zielperson"""
    treffer = alle[alle["label"].fillna("").str.contains(begriff, case=False, regex=False)]
    if kohorte:
        treffer = treffer[treffer["kohorte"] == kohorte]
    if quelle:
        treffer = treffer[treffer["var"].str.startswith(quelle)]
    return treffer[["kohorte", "datensatz", "var", "label"]].reset_index(drop=True)


# hier kann man nach Variablen und Labels suchen, z. B. nach "Abschluss" in den Angaben der Eltern

suche("Bundesland", quelle="p")

,kohorte,datensatz,var,label
0,SC1,Demography,p751001_g2,Wohnort (Bundesland)


In [ ]:
## Anzahl der IDs pro Kohorte und Datensatz prüfen


In [12]:
# Daten einer Variablen auslesen und prüfen, wie viele gültige Werte pro Welle vorliegen

def pruefe(kohorte, datensatz, var):
    pfad = next(data_dir.rglob(f"{kohorte}_NEPScomp_{datensatz}_*.sav"))
    df, meta = pyreadstat.read_sav(pfad, usecols=["ID_t", "wave", var])
    print("Label:", meta.column_names_to_labels[var])
    print("Wertelabels:", meta.variable_value_labels.get(var))
    print("\nGültige Werte pro Welle:")
    print(df.groupby("wave")[var].count())
    return df

# Beispiel, Namen anpassen:
# _ = pruefe("SC3", "Competency", "ma_sc1")
pruefe("SC4", "Demography", "p510008")  # monatl. Haushaltseinkommen, Klassen 2.000 - 4.000 Euro

Label: monatl. Haushaltseinkommen, Klassen 2.000 - 4.000 Euro
Wertelabels: {-98.0: 'weiß nicht', -97.0: 'Angabe verweigert', -54.0: 'designbedingt fehlend', 4.0: '2.000 bis unter 2.500 Euro', 5.0: '2.500 bis unter 3.000 Euro', 6.0: '3.000 bis unter 4.000 Euro'}

Gültige Werte pro Welle:
wave
1.0       0
2.0       0
3.0       0
4.0       0
5.0     128
6.0       0
7.0     107
8.0       0
9.0       0
10.0      0
11.0      0
12.0      0
13.0      0
14.0      0
15.0      0
16.0      0
Name: p510008, dtype: int64


,ID_t,wave,p510008
0,4010908.0,1.0,NaN
1,4010908.0,2.0,NaN
2,4010908.0,3.0,NaN
3,4010908.0,4.0,NaN
4,4010908.0,5.0,NaN
...,...,...,...
262795,4037069.0,12.0,NaN
262796,4037069.0,13.0,NaN
262797,4037069.0,14.0,NaN
262798,4037069.0,15.0,NaN


In [13]:
kandidaten = ["p724111", "p724112", "p725000", "p731802_g1", "p510006_g1",
              "p741001", "p731901", "p731951", "p400500_g3", "p400000_g1",
              "p700010", "p732103", "p751001_g1", "p731110", "p412000"]

def abdeckung(kohorte, var):
    treffer = alle[(alle["kohorte"] == kohorte) & (alle["var"] == var)]
    if treffer.empty:
        print(f"nicht in NEPScomp {kohorte}: {var}")
        return None
    ds = treffer["datensatz"].iloc[0]
    pfad = next(data_dir.rglob(f"{kohorte}_NEPScomp_{ds}_*.sav"))
    df, _ = pyreadstat.read_sav(pfad, usecols=["ID_t", "wave", var])
    return df.groupby("wave")[var].count().rename(var)

serien = [abdeckung("SC1", v) for v in kandidaten]
tab = pd.concat([s for s in serien if s is not None], axis=1)
tab

,p724111,p724112,p725000,p731802_g1,p510006_g1,p741001,p731901,p731951,p400500_g3,p400000_g1,p700010,p732103,p751001_g1,p731110,p412000
wave,,,,,,,,,,,,,,,
1.0,0,0,0,3472,3093,3480,3477,3234,3481,3481,3481,0,3481,3478,1420
2.0,0,0,0,2838,2563,2846,58,53,3481,3481,3481,0,58,0,0
3.0,0,0,0,2608,2387,2608,2609,2426,3481,3481,3481,0,7,2609,807
4.0,0,0,0,2477,2268,2478,2476,2274,3481,3481,3481,2478,2477,2478,2365
5.0,0,0,0,2379,2214,2381,2380,2175,3481,3481,3481,0,2379,2381,2318
6.0,0,0,0,2206,2043,2207,2209,2013,3481,3481,3481,0,2209,2207,2147
7.0,0,0,0,2116,1965,2114,2116,1908,3481,3481,3481,0,2115,2115,2055
8.0,0,0,33,2069,1918,2068,2070,1857,3481,3481,3481,2070,2069,2069,2024
9.0,0,0,1794,1847,1738,1846,1846,1636,3481,3481,3481,0,1847,1847,1784


In [17]:
neu = ["p731904_g14", "p731954_g14", "p731852_g1", "p751001_g2", "p731802_g1"]
serien = [abdeckung("SC1", v) for v in neu]
display(pd.concat([s for s in serien if s is not None], axis=1))

for v in ["p731802_g1", "p731852_g1", "p751001_g2"]:
    ds = alle[(alle["kohorte"] == "SC1") & (alle["var"] == v)]["datensatz"].iloc[0]
    pfad = next(data_dir.rglob(f"SC1_NEPScomp_{ds}_*.sav"))
    _, meta = pyreadstat.read_sav(pfad, metadataonly=True)
    print(v, meta.variable_value_labels.get(v), "\n")

,p731904_g14,p731954_g14,p731852_g1,p751001_g2,p731802_g1
wave,,,,,
1.0,3142,2981,3224,3481,3472
2.0,51,43,2663,58,2838
3.0,1324,461,2461,7,2608
4.0,504,1758,2284,2477,2477
5.0,461,347,2279,2379,2379
6.0,392,207,2129,2209,2206
7.0,357,194,2041,2115,2116
8.0,288,220,2002,2069,2069
9.0,274,160,1792,1847,1847


p731802_g1 {-55.0: 'nicht ermittelbar', -54.0: 'designbedingt fehlend', -20.0: 'kein Schulabschluss', 0.0: '[0A/1A] kein Abschluss', 1.0: '[2B] Haupt-, Volksschulabschluss, Berufsvorbereitende Maßnahme', 2.0: '[2A] Mittlere Reife, Realschulabschluss', 3.0: '[3A] Fachhochschulreife, Hochschulreife', 4.0: '[3B] Lehre, Berufsfachsch., Fachsch. des Gesundheitswesens (weniger als zwei Jahre), Beamter einfacher Dienst, berufliche Grundkenntnisse', 5.0: '[3C] Beamter mittlerer Dienst', 6.0: '[4A] Fachhochschulreife, Hochschulreife (second cycle)', 7.0: '[4B] Lehre, Berufsfachschule, Fachschule des Gesundheitswesens (weniger als zwei Jahre), Beamter einfacher Dienst, berufliche Grundkenntnisse (second cycle)', 8.0: '[5B] Fach- und Berufsakad. Abschl., Verwaltungsfachhochsch., Fachsch. des Gesundheitswesens (mind. zwei Jahre), Meister/Techniker, anderer Fachschulabschl., Beamter gehobener Dienst', 9.0: '[5A] Bachelor, Master, Diplom, Magister, Staatsexamen, Beamter höherer Dienst', 10.0: '[6] P

In [18]:
# Objektive Mathe-Kompetenz in SC1 (laut Manual Wellen 5, 7, 9, 11)
display(abdeckung("SC1", "ma_sc1"))

# Schulform bzw. Bildungsstatus in SC1
for begriff in ["Schulart", "Schulform", "Gymnasium", "Übergang"]:
    print("\n====", begriff)
    display(suche(begriff, kohorte="SC1"))

wave
1.0        0
2.0        0
3.0        0
4.0        0
5.0     1683
6.0        0
7.0     1597
8.0        0
9.0     1240
10.0       0
11.0    1315
12.0       0
Name: ma_sc1, dtype: int64


==== Schulart


,kohorte,datensatz,var,label



==== Schulform


,kohorte,datensatz,var,label
0,SC1,Resources,p66600a,Schulform realistisch
1,SC1,Resources,p66601a,Schulform idealistisch



==== Gymnasium


,kohorte,datensatz,var,label



==== Übergang


,kohorte,datensatz,var,label
